In [ ]:
import sys 
sys.path.append('..')

import pickle
import pandas as pd

from scripts.global_vars import (
    N_TRIALS, 
    SAMPLE_SIZE, 
    COLUMN_NAME,
    USE_TRAIN_CHECKPOINT
)

from scripts.pipeline.nmf_pipeline import nmf_pipeline, inference_pipeline
from scripts.pipeline.visualization import plot_topics_wordcloud, plot_topics_cluster

---
### **STAGE 1. PREPROCESSING**
---

In [ ]:
df = pd.read_csv("../data/corpus/topic_dataset.csv")
sample_df = df.sample(n=min(SAMPLE_SIZE, len(df)), random_state=42)
sample_df

In [ ]:
corpus = sample_df[COLUMN_NAME].tolist()
corpus

---
### **STAGE 2. MODEL EXPERIMENTS**
---

In [ ]:
vectorizer_params = {
    'ngram_range': {'type': 'int', 'low': 2, 'high': 4},
    'max_features': {'type': 'int', 'low': 2000, 'high': 15000},
    'min_df': {'type': 'float', 'low': 0.001, 'high': 0.2},
    'max_df': {'type': 'float', 'low': 0.75, 'high': 0.99},
    'stop_words': {'type': 'categorical', 'choices': ['english', None]}
}

nmf_params = {
    'n_components': {'type': 'int', 'low': 5, 'high': 30},
    # 'l1_ratio': {'type': 'float', 'low': 0.0, 'high': 0.8},
    # 'max_iter': {'type': 'int', 'low': 300, 'high': 800}
}

In [ ]:
if USE_TRAIN_CHECKPOINT:
    topic_model = pickle.load(open(f"../data/models/nmf_model_{COLUMN_NAME}.pkl", "rb"))
    topics = pickle.load(open(f"../data/inference/nmf_inference_{COLUMN_NAME}.pkl", "rb"))

else:
    topic_model = nmf_pipeline(
        corpus, 
        vectorizer_params=vectorizer_params, 
        nmf_params=nmf_params, 
        n_trials=N_TRIALS, 
        n_jobs=10, 
        save=f"../data/models/nmf_model_{COLUMN_NAME}.pkl"
    )
    topics, probs = inference_pipeline(
        topic_model, 
        corpus, 
        save=f"../data/inference/nmf_inference_{COLUMN_NAME}.pkl"
    )

In [ ]:
sample_df['topic'] = topics
sample_df['prob'] = probs.max(axis=1)

sample_df

In [ ]:
topics = topic_model.get_topics()
topic_info = topic_model.get_topic_info()

topic_info

---
### **STAGE 3. VISUALIZE ALMOST ANYTHING!**
---

In [ ]:
plot_topics_cluster(sample_df, topic_model, 20)

In [ ]:
topics[-1] = {}
plot_topics_wordcloud(topics, topic_model)